# orderflow-lm: injection experiments on Colab

Run top to bottom. Everything durable lives on Drive under `DRIVE`; the VM disk is scratch.
Injection runs save per-batch parts, so after a disconnect, re-run the setup cells and then
the same run cell: finished batches are skipped.

Prerequisite: upload the paper checkpoint (`v2.pt`, step 22,000) to `DRIVE/checkpoints/v2.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/oflm'      # durable: checkpoints, test days, results
BRANCH = 'icml-week0'
import os
for d in ('checkpoints', 'normalized', 'test_days', 'results'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)

In [ ]:
%cd /content
!rm -rf orderflow-lm && git clone -q -b {BRANCH} https://github.com/KookiesNKareem/orderflow-lm.git
%cd /content/orderflow-lm
# keep Colab's CUDA build of torch: install the package without dependencies
!pip -q install --no-deps -e . && pip -q install polars pyarrow requests scikit-learn
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Checkpoint
Copied to local disk and checked against the hash recorded for the paper's final injection run.

In [ ]:
import hashlib, json, shutil
CKPT = '/content/v2.pt'
shutil.copy(f'{DRIVE}/checkpoints/v2.pt', CKPT)
want = json.load(open('results/paper/injection/run_metadata.json'))['checkpoint_sha256']
h = hashlib.sha256(open(CKPT, 'rb').read()).hexdigest()
print('sha256 ok' if h == want else f'MISMATCH: {h} != {want}')

## Test days

Walk-forward test days after the training cutoff. Each day is downloaded from Tardis (free first-of-month files),
reconstructed, tokenized into its own file, and cached on Drive; days already on Drive are skipped.
`2026-07-01` is the paper's holdout day (use `--span 0.5,1` for its test half).

In [ ]:
SYMBOLS = ['SOLUSDT', 'ETHUSDT', 'BTCUSDT']
DATES = ['2026-07-01', '2026-08-01', '2026-09-01', '2026-10-01']
months = sorted({d[:7] for d in DATES})

# normalized parquet is the slow step; build on the VM, then cache on Drive
!mkdir -p data && rm -rf data/normalized && ln -s {DRIVE}/normalized data/normalized
for m in months:
    !python scripts/build_corpus.py --symbols {' '.join(SYMBOLS)} --start {m} --end {m} --workers 3
!python scripts/make_test_days.py --norm data/normalized --out {DRIVE}/test_days --dates {' '.join(DATES)} --symbols {' '.join(SYMBOLS)}

## Generation benchmark

Throughput decides how much of the multi-model, multi-day plan fits on Colab. One batch of each size,
all five arms, on SOL's paper test day.

In [ ]:
import time, subprocess
for B in (64, 128, 256):
    out = f'/content/bench_b{B}'
    t0 = time.time()
    subprocess.run(['python', 'scripts/pilot_inject.py', '--bin', f'{DRIVE}/test_days/SOLUSDT_2026-07-01.bin',
                    '--ckpt', CKPT, '--symbol', 'SOLUSDT', '--seeds', str(B), '--batch', str(B),
                    '--new-tokens', '2500', '--pcrn', '--span', '0.5,1', '--out', out], check=True,
                   stdout=subprocess.DEVNULL)
    dt = time.time() - t0
    print(f'batch {B}: {dt:.0f}s for 5 arms -> {5 * B * 2500 / dt:,.0f} tok/s; '
          f'2,048 windows ~ {dt * 2048 / B / 60:.0f} min per symbol-day')

In [ ]:
# smoke-test the new analysis scripts on the benchmark output before any long run
!python scripts/eval_context_split.py --pilot /content/bench_b64 --symbol SOLUSDT
!python scripts/eval_depth_response.py --pilot /content/bench_b64 --symbol SOLUSDT

## Injection run

One symbol-day per call; outputs go straight to Drive. Re-running after a disconnect resumes from saved parts.

In [ ]:
def inject(symbol, date, seeds=2048, batch=128, span='0,1'):
    out = f'{DRIVE}/results/inject/{date}'
    !python scripts/pilot_inject.py --bin {DRIVE}/test_days/{symbol}_{date}.bin --ckpt {CKPT} \
        --symbol {symbol} --seeds {seeds} --batch {batch} --new-tokens 2500 --pcrn --span {span} --out {out}
    return out

out = inject('SOLUSDT', '2026-07-01', span='0.5,1')   # reproduces the paper's SOL run on rebuilt data

## Week-0 analyses

Kernel and trade-clock response, causal statistics, the context-flow split, and depth depletion in the first generated frame.

In [ ]:
def analyze(symbol, date):
    out, tape = f'{DRIVE}/results/inject/{date}', f'{DRIVE}/test_days/{symbol}_{date}.bin'
    !python scripts/eval_propagator.py --bin {tape} --symbol {symbol} --pilot {out}
    !python scripts/eval_causal_stats.py --pilot {out} --symbol {symbol}
    !python scripts/eval_context_split.py --pilot {out} --symbol {symbol} --bin {tape}
    !python scripts/eval_depth_response.py --pilot {out} --symbol {symbol} --bin {tape}

analyze('SOLUSDT', '2026-07-01')